In [42]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
import scipy.sparse as sp

## SN: Bhat-Nakshatri et al
Link: ?

In [43]:
# Load the .h5ad file
adata_sn = sc.read_h5ad("breast_sn.h5ad")

In [44]:
adata_sn.obs["cell_type"].value_counts()

cell_type
basal-myoepithelial cell of mammary gland                     16184
luminal adaptive secretory precursor cell of mammary gland    11456
fibroblast                                                     6526
endothelial cell                                               5407
luminal hormone-sensing cell of mammary gland                  5202
adipocyte                                                      2712
T cell                                                         2240
macrophage                                                     1640
Name: count, dtype: int64

In [45]:
adata_sn = adata_sn[~adata_sn.obs['cell_type'].isin(['adipocyte', 
                                      'luminal hormone-sensing cell of mammary gland', 
                                      'luminal adaptive secretory precursor cell of mammary gland'])]

In [46]:
GEX_dict_ct = {
    'basal-myoepithelial cell of mammary gland': 'basal cell',
    'fibroblast': 'fibroblast',
    'endothelial cell': 'endothelial cell',
    'adipocyte': 'adipocyte',
    'T cell': 'T cell',
    'macrophage': 'macrophage'
}

In [47]:
cell_type = [GEX_dict_ct[ct] for ct in adata_sn.obs['cell_type']]
adata_sn.obs['cell_type'] = cell_type
adata_sn.obs['cell_type'] = adata_sn.obs['cell_type'].astype('category')

/tmp/ipykernel_4083039/439307345.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sn.obs['cell_type'] = cell_type


In [48]:
adata_sn.obs['cell_type'].value_counts()

cell_type
basal cell          16184
fibroblast           6526
endothelial cell     5407
T cell               2240
macrophage           1640
Name: count, dtype: int64

## SC: Kumar et al
Link: ?

In [49]:
# Load the .h5ad file
adata_sc = sc.read_h5ad("breast_sc.h5ad")

In [63]:
adata_sc.obs["cell_type"].value_counts()

cell_type
fibroblast          195127
basal cell           84377
T cell               72746
endothelial cell     32071
macrophage            1293
Name: count, dtype: int64

In [51]:
adata_sc = adata_sc[adata_sc.obs['assay']=="10x 3' v3"]

In [52]:
adata_sc = adata_sc[(adata_sc.obs['broad_cell_type'].isin(['basal', 'fibroblasts', 'tcells']))|
          (adata_sc.obs['cell_type'].isin(['vein endothelial cell', 'macrophage']))
         ]

In [53]:
adata_sc.obs['broad_cell_type'].value_counts()

broad_cell_type
fibroblasts    195127
basal           84377
tcells          72746
vascular        32071
myeloid          1293
Name: count, dtype: int64

In [54]:
RNA_dict_ct = {
    'fibroblasts' : 'fibroblast',
    'basal' : 'basal cell',
    'tcells' : 'T cell',
    'vascular' : 'endothelial cell',
    'myeloid' : 'macrophage'
}

In [55]:
cell_type = [RNA_dict_ct[ct] for ct in adata_sc.obs['broad_cell_type']]
adata_sc.obs['cell_type'] = cell_type
adata_sc.obs['cell_type'] = adata_sc.obs['cell_type'].astype('category')

/tmp/ipykernel_4083039/1097223198.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sc.obs['cell_type'] = cell_type


In [56]:
adata_sc.obs['cell_type'].value_counts()

cell_type
fibroblast          195127
basal cell           84377
T cell               72746
endothelial cell     32071
macrophage            1293
Name: count, dtype: int64

## Processing

In [57]:
print("Max value gene_exp:", adata_sn.X.max())
print("Max value gene_exp:", adata_sc.X.max())

Max value gene_exp: 6.7707896
Max value gene_exp: 8.655264


In [58]:
import numpy as np, scipy.sparse as sp

def is_loglike(adata, name):
    X = adata.X
    vals = np.asarray(X.data) if sp.issparse(X) else np.ravel(X)
    print(f"{name}: max={X.max():.3f}, any>50={bool((vals>50).any())}, q99.9={np.quantile(vals,0.999):.3f}")

is_loglike(adata_sn, "sn (GEX)")
is_loglike(adata_sc, "sc (RNA)")

sn (GEX): max=6.771, any>50=False, q99.9=3.401
sc (RNA): max=8.655, any>50=False, q99.9=5.060


In [59]:
# Make the intersection of the cell types in the two datasets
cell_types_sn = set(adata_sn.obs["cell_type"].unique())
cell_types_sc = set(adata_sc.obs["cell_type"].unique())
common_cell_types = cell_types_sn.intersection(cell_types_sc)
common_cell_types

{'T cell', 'basal cell', 'endothelial cell', 'fibroblast', 'macrophage'}

In [ ]:
# Create a df table to show me how many cells of each cell type are in the two datasets, just for the intersection cell types
import pandas as pd
cell_type_counts_sn = adata_sn.obs["cell_type"].value_counts()
cell_type_counts_sc = adata_sc.obs["cell_type"].value_counts()
df_counts = pd.DataFrame({
    "cell_type": list(common_cell_types),
    "count_sn": [cell_type_counts_sn[cell_type] for cell_type in common_cell_types],
    "count_sc": [cell_type_counts_sc[cell_type] for cell_type in common_cell_types]
})

df_counts["min_count"] = df_counts[["count_sn", "count_sc"]].min(axis=1)
df_counts

In [64]:
import numpy as np
import pandas as pd

def balance_two_adatas_by_celltype(
    adata_a,
    adata_b,
    celltype_col="cell_type",
    celltypes=None,
    random_state=0,
    force_targets=None,
):
    rng = np.random.default_rng(random_state)

    # pick shared cell types if not provided
    if celltypes is None:
        celltypes = sorted(set(adata_a.obs[celltype_col]).intersection(set(adata_b.obs[celltype_col])))

    counts_a = adata_a.obs[celltype_col].value_counts()
    counts_b = adata_b.obs[celltype_col].value_counts()

    # min per cell type
    min_counts = {}
    for ct in celltypes:
        if ct in counts_a and ct in counts_b:
            min_counts[ct] = int(min(counts_a[ct], counts_b[ct]))

    # override targets if requested (clip to what's available in BOTH datasets)
    if force_targets:
        for ct, k in force_targets.items():
            if ct in min_counts:
                min_counts[ct] = int(min(k, counts_a[ct], counts_b[ct]))

    def subset_to_targets(adata, targets):
        idx_keep = []
        for ct, k in targets.items():
            idx_ct = adata.obs_names[adata.obs[celltype_col] == ct].to_numpy()
            if len(idx_ct) <= k:
                idx_keep.append(idx_ct)
            else:
                idx_keep.append(rng.choice(idx_ct, size=k, replace=False))
        idx_keep = np.concatenate(idx_keep) if len(idx_keep) else np.array([], dtype=object)
        return adata[idx_keep].copy()

    adata_a_bal = subset_to_targets(adata_a, min_counts)
    adata_b_bal = subset_to_targets(adata_b, min_counts)

    summary = pd.DataFrame({
        "cell_type": list(min_counts.keys()),
        "count_a_before": [int(counts_a[ct]) for ct in min_counts.keys()],
        "count_b_before": [int(counts_b[ct]) for ct in min_counts.keys()],
        "target_count":   [int(min_counts[ct]) for ct in min_counts.keys()],
    }).sort_values("cell_type").reset_index(drop=True)

    return adata_a_bal, adata_b_bal, summary

In [65]:
adata_sn_bal, adata_sc_bal, summary = balance_two_adatas_by_celltype(
    adata_sn, adata_sc,
    celltype_col="cell_type",
    celltypes=common_cell_types,
    random_state=42,
    force_targets=None
)

In [ ]:
adata_sn_bal.obs["cell_type"].value_counts()

In [ ]:
adata_sc_bal.obs["cell_type"].value_counts()

In [66]:
X_sn = adata_sn_bal.X.toarray()
X_sc = adata_sc_bal.X.toarray()

obs_sn = pd.DataFrame(adata_sn_bal.obs['cell_type'])
var_sn = pd.DataFrame(adata_sn_bal.var['feature_name'])

obs_sc = pd.DataFrame(adata_sc_bal.obs['cell_type'])
var_sc = pd.DataFrame(adata_sc_bal.var['feature_name'])

In [67]:
adata_sn_final = ad.AnnData(X=X_sn, obs=obs_sn, var=var_sn)
adata_sc_final = ad.AnnData(X=X_sc, obs=obs_sc, var=var_sc)

In [68]:
print("Max value gene_exp:", adata_sn_final.X.max())
print("Max value gene_exp:", adata_sc_final.X.max())

Max value gene_exp: 6.7707896
Max value gene_exp: 8.371428


In [72]:
# Save
adata_sn_final.write_h5ad("breast_sn_filt.h5ad", compression="gzip")
adata_sc_final.write_h5ad("breast_sc_filt.h5ad", compression="gzip")

## Analyze fake data

In [ ]:
adata_fake = sc.read_h5ad("breast_Fake_RNA_VAE_UNET_log_norm.h5ad")

In [ ]:
adata_sc_final.var.feature_name

In [ ]:
adata_fake.var.feature_name

In [ ]:
print(f"sc max value: {adata_sc_final.X.max()}")

In [ ]:
print(f"sc max value: {adata_fake.X.max()}")